# Components lab 03: classical and quantum channels

Channels are components in the middle of a port graph. They receive a port delivery, apply transport behavior, and schedule a new delivery through their output port.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from simyuj.components import (
    ACTION_RECEIVE_CLASSICAL,
    ACTION_TRANSMIT_CLASSICAL,
    ACTION_TRANSMIT_QUANTUM,
    ClassicalChannel,
    Port,
    PortDelivery,
    PortDirection,
    PortKind,
    QuantumChannel,
    connect_ports,
)
from simyuj.engine import Component, Timeline
from simyuj.primitives.messages import ClassicalMessage
from simyuj.primitives.subsystems import SubsystemHandle
from simyuj.qstate import SubsystemId
from simyuj.qstate.noise import depolarizing
from simyuj.qstate.state import purity
from simyuj.runtime.binding import BindingContext
from simyuj.signal import EncodingScheme, Signal, SignalKind

## 1. Lab endpoints

The endpoints do not model physics. They let the real channel components sit in the middle.

In [ ]:
@dataclass(slots=True)
class ClassicalSender(Component):
    component_id: str
    output_port: Port = field(init=False)

    def __post_init__(self) -> None:
        self.output_port = Port('out', self, self.component_id, PortKind.CLASSICAL, PortDirection.EGRESS)

    def handle_event(self, event, timeline) -> None:
        raise ValueError(event.action)


@dataclass(slots=True)
class ClassicalInbox(Component):
    component_id: str
    input_port: Port = field(init=False)
    received: list[tuple[int, ClassicalMessage]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port('in', self, self.component_id, PortKind.CLASSICAL, PortDirection.INGRESS)

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        self.received.append((timeline.current_time, delivery.payload))


In [ ]:
@dataclass(slots=True)
class QuantumSender(Component):
    component_id: str
    output_port: Port = field(init=False)

    def __post_init__(self) -> None:
        self.output_port = Port('out', self, self.component_id, PortKind.QUANTUM, PortDirection.EGRESS)

    def handle_event(self, event, timeline) -> None:
        raise ValueError(event.action)


@dataclass(slots=True)
class QuantumInbox(Component):
    component_id: str
    input_port: Port = field(init=False)
    received: list[tuple[int, Signal]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port('in', self, self.component_id, PortKind.QUANTUM, PortDirection.INGRESS)

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        self.received.append((timeline.current_time, delivery.payload))


## 2. Classical channel: distance gives delay, loss drops messages

The channel must be bound before execution so it can declare its loss RNG stream.

In [ ]:
timeline = Timeline(master_seed=31)
alice = ClassicalSender('alice.ctrl')
bob = ClassicalInbox('bob.ctrl')
channel = ClassicalChannel(
    channel_id='fiber.classical.12km',
    length_m=12_000,
    loss_probability=0.30,
)
channel.bind(BindingContext(timeline=timeline, logger=timeline.logger))

connect_ports(alice.output_port, channel.input_port, target_action=ACTION_TRANSMIT_CLASSICAL)
connect_ports(channel.output_port, bob.input_port, target_action=ACTION_RECEIVE_CLASSICAL)

print('resolved delay ticks:', channel.resolved_delay_ticks)
print('loss probability:', channel.loss_probability)

In [ ]:
for round_id in range(1, 7):
    message = ClassicalMessage(
        sender_id='alice',
        receiver_id='bob',
        body=f'basis announcement {round_id}',
        sent_time=round_id,
        message_type='basis.announce',
        round_id=round_id,
    )
    alice.output_port.connection.transmit(message, timeline, time=round_id)

timeline.run_until_empty()

print('channel counts:', {
    'received': channel.received_count,
    'delivered': channel.delivered_count,
    'dropped': channel.dropped_count,
})
print('bob received:')
for time, message in bob.received:
    print(' t=', time, '|', message.body, '| sent at', message.sent_time)

## 3. Build the `Signal` before the quantum channel sees it

A quantum channel carries a `Signal`. The qstate lives in the timeline store; the signal tells later components which state reference and subsystem handle to preserve.


In [ ]:
def make_signal(timeline: Timeline, *, signal_id: str, state: str, origin: str, tick: int) -> Signal:
    subsystem = SubsystemId(f'{signal_id}:qubit')
    state_ref = timeline.qstate.prepare(state, rep='density', subsystems=(subsystem,))
    return Signal(
        id=signal_id,
        signal_kind=SignalKind.PHOTON,
        encoding_scheme=EncodingScheme.POLARIZATION,
        emission_time=tick,
        origin=origin,
        state_ref=state_ref,
        state_targets=(SubsystemHandle(label=str(subsystem), kind='qubit', index=0),),
        protocol_params=(('basis_hint', 'Z-or-X'),),
        meta=(('wavelength_nm', 1550.0), ('source_clock', tick)),
        timing_meta=(('launched_tick', tick),),
    )


In [ ]:
preview_timeline = Timeline(master_seed=41)
preview = make_signal(
    preview_timeline,
    signal_id='preview-photon',
    state='|+>',
    origin='alice.qtx',
    tick=12,
)

print('signal id:', preview.id)
print('kind and encoding:', preview.signal_kind.value, preview.encoding_scheme.value)
print('emission tick:', preview.emission_time)
print('state_ref:', preview.state_ref)
print('state_targets:', preview.state_targets)
print('protocol params:', dict(preview.protocol_params))
print('meta:', dict(preview.meta))


## 4. Quantum channel: survival, jitter, and qstate noise

Now send several signals through a noisy fiber. Some are lost, and survivors keep enough information for the receiver to inspect the qstate record.


In [ ]:
timeline = Timeline(master_seed=44)
source = QuantumSender('alice.qtx')
receiver = QuantumInbox('bob.qrx')
qchannel = QuantumChannel(
    channel_id='fiber.quantum.8km',
    length_m=8_000,
    attenuation_db_per_km=0.20,
    fixed_insertion_loss_db=0.40,
    timing_jitter_stddev_ticks=2.0,
    noise_models=(depolarizing(0.12),),
)
qchannel.bind(BindingContext(timeline=timeline, logger=timeline.logger))

connect_ports(source.output_port, qchannel.input_port, target_action=ACTION_TRANSMIT_QUANTUM)
connect_ports(qchannel.output_port, receiver.input_port, target_action='receive_signal')

print('base delay ticks:', qchannel.resolved_delay_ticks)
print('survival probability:', round(qchannel.survival_probability, 3))

In [ ]:
for index, state in enumerate(['|0>', '|+>', '|1>', '|+>', '|0>'], start=1):
    signal = make_signal(
        timeline,
        signal_id=f'photon-{index}',
        state=state,
        origin='alice.qtx',
        tick=index * 2,
    )
    source.output_port.connection.transmit(signal, timeline, time=index * 2)

timeline.run_until_empty()

print('quantum channel counts:', {
    'received': qchannel.received_count,
    'delivered': qchannel.delivered_count,
    'lost': qchannel.lost_count,
})
print('live qstate records after loss:', timeline.qstate.size())

In [ ]:
print('received quantum signals:')
for time, signal in receiver.received:
    print(' t=', time, '| id=', signal.id)
    print('   timing:', dict(signal.timing_meta))
    record = timeline.qstate.record(signal.state_ref)
    print('   rep:', record.rep, 'density diagonal:', np.round(np.diag(record.payload.rho).real, 3).tolist())
    print('   purity:', round(purity(record.payload), 3))

## Keep this model in your head

A classical channel preserves message contents but may delay or drop them. A quantum channel carries a `Signal`: the envelope stays readable while qstate storage remains owned by the timeline. Lost signals remove targets; survivors may pick up timing metadata and qstate noise.
